# quantumQuery

# Part 1: Setup

In [ ]:
import sys
import json

sys.path.insert(0, "src")

import numpy as np
import pandas as pd

import corpus_retrieval as cr
import llm_experiment as lx

REBUILD = False
USE_DENSE = True
LLM_BACKEND = "ollama"
MANUAL_CHECK = False

NEED_LLM = REBUILD or MANUAL_CHECK or not (lx.EVAL_AUTO_JSON.exists() and lx.RESULTS_JSON.exists())
cr.environment_report()

# Part 2: Corpus

In [ ]:
if REBUILD or not cr.CORPUS_CSV.exists():
    cr.build_corpus()

corpus = pd.read_csv(cr.CORPUS_CSV)
stats = json.loads(cr.CORPUS_STATS.read_text())
print(len(corpus), "papers")
pd.Series(stats["domain_counts"], name="papers").to_frame()

# Part 3: Encoder and embeddings

In [ ]:
needs_encoding = REBUILD or not cr.EMBEDDINGS_NPY.exists()
encoder = cr.load_encoder() if (USE_DENSE or needs_encoding) else None

if needs_encoding:
    cr.encode_corpus(corpus, encoder)

embeddings = np.load(cr.EMBEDDINGS_NPY)
cr.verify_artifacts()

# Part 4: Retrieval and LLM layer

In [ ]:
indices = cr.build_indices(corpus, embeddings, encoder if USE_DENSE else None)
client = lx.make_llm_client(LLM_BACKEND) if NEED_LLM else None

if NEED_LLM:
    lx.check_backend(LLM_BACKEND)

stored = json.loads(lx.RESULTS_JSON.read_text())["disagreement_examples"][0] if lx.RESULTS_JSON.exists() else None
if stored:
    lx.show_example(corpus, indices, stored)

# Part 5: Evaluation set and experiment

In [ ]:
if REBUILD or not lx.EVAL_AUTO_JSON.exists():
    lx.build_eval_set(corpus, client)

eval_set = json.loads(lx.EVAL_AUTO_JSON.read_text())

if REBUILD or not lx.RESULTS_JSON.exists():
    lx.run_experiment(corpus, indices, eval_set, client)

results = json.loads(lx.RESULTS_JSON.read_text())
lx.summary_table(results)

# Part 6: Analysis and manual check

In [ ]:
analysis = lx.run_analysis(corpus, embeddings, eval_set, results, indices)
lx.print_summary(analysis)

In [ ]:
lx.plot_all(analysis)

In [ ]:
if MANUAL_CHECK:
    lx.manual_check(corpus, indices, client)

lx.manual_summary()